# AUModel — Phase 2: Model Architecture

This notebook verifies the AUModel (~749.5M parameter decoder-only transformer) on a Colab GPU.

| Section | Action | Est. Time |
|---------|--------|-----------|
| 1 | GPU/BF16 setup + PyTorch ≥ 2.1 check | < 1 min |
| 2 | Mount Drive + clone repo | 1–2 min |
| 3 | Load model → float32 sanity checks | 2–3 min |
| 4 | 4-check sanity block (shape, params, loss) | < 1 min |
| 5 | Single-batch overfit (50 AdamW steps, loss → < 0.1) | 2–4 min |
| 6 | Clean-batch inference check | < 1 min |
| 7 | Results summary | — |

---
## Cell 1: GPU / BF16 Setup + Dependency Check

In [ ]:
# Cell 1: GPU/BF16 setup + dependency check
# Expected: CUDA available, PyTorch >= 2.1, bfloat16 supported
import sys
import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")

# Check PyTorch >= 2.1
major, minor = (int(x) for x in torch.__version__.split('.')[:2])
if (major, minor) < (2, 1):
    raise RuntimeError(f"PyTorch >= 2.1 required, got {torch.__version__}")
print("[OK] PyTorch >= 2.1")

# Check CUDA
if not torch.cuda.is_available():
    raise RuntimeError("GPU not available — go to Runtime > Change runtime type and select T4 GPU")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"[OK] GPU: {gpu_name} ({total_gb:.1f} GB VRAM)")

# Check bfloat16 support
if not torch.cuda.is_bf16_supported():
    print("[WARN] BF16 not natively supported on this GPU — proceeding with float32 for sanity checks")
    USE_BF16 = False
else:
    print("[OK] BF16 supported")
    USE_BF16 = True

# Perf flags (constitution)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print("[OK] TF32 enabled for matmul + cuDNN")

---
## Cell 2: Mount Drive + Clone Repo

In [ ]:
# Cell 2: Mount Google Drive + clone/update repo
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT_DIR = '/content/drive/MyDrive/aumodel_checkpoints'
MODEL_CKPT_DIR = f'{DRIVE_CKPT_DIR}/model'

import os
os.makedirs(MODEL_CKPT_DIR, exist_ok=True)
print(f"[OK] Drive mounted. Model checkpoints → {MODEL_CKPT_DIR}")

# Clone / pull repo
REPO_URL    = 'https://github.com/aykanugur/AU_MODEL.git'
REPO_BRANCH = '002-model-architecture'
REPO_DIR    = '/content/AU_MODEL'

if not os.path.exists(REPO_DIR):
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    print("[OK] Repo cloned")
else:
    print("[OK] Repo already present — pulling latest")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
!git log --oneline -3

# Add repo root to sys.path so `import model` works
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"[OK] Working dir: {os.getcwd()}")

---
## Cell 3: Load Model → GPU + BF16

In [ ]:
# Cell 3: Instantiate AUModel on GPU with bfloat16
from model import AUModel, ModelConfig

cfg   = ModelConfig()
model = AUModel(cfg)

# Move to GPU + cast to bfloat16 (or float32 fallback)
dtype = torch.bfloat16 if USE_BF16 else torch.float32
model = model.to(device).to(dtype)
print(f"[OK] Model moved to {device} in {dtype}")

# Verify freqs_cis buffer also moved
buf_device = model.freqs_cis.device
if str(buf_device) != str(device):
    raise RuntimeError(f"freqs_cis buffer not on GPU: {buf_device}")
print(f"[OK] freqs_cis buffer on {buf_device}")

n_params = model.get_num_params()
print(f"[OK] Param count: {n_params:,} ({n_params / 1e6:.1f}M)")

# VRAM usage after loading
allocated_gb = torch.cuda.memory_allocated(0) / 1e9
print(f"[OK] VRAM allocated: {allocated_gb:.2f} GB")

---
## Cell 4: 4-Check Sanity Block (Mirrors `model/sanity_check.py`)

In [ ]:
# Cell 4 (Spec: Cell 3) — 4-check sanity block
# Mirrors model/sanity_check.py but runs in-notebook on GPU
import math
import torch.nn.functional as F

results = {}

# --- Check S1: forward shape ---
model.eval()
B, T = 2, 128
tokens = torch.randint(0, cfg.vocab_size, (B, T), device=device)
with torch.no_grad():
    logits, _ = model(tokens)

expected_shape = (B, T, cfg.vocab_size)
if logits.shape == expected_shape:
    print(f"[PASS] S1 shape: {tuple(logits.shape)}")
    results['S1_shape'] = True
else:
    print(f"[FAIL] S1 shape: got {tuple(logits.shape)}, expected {expected_shape}")
    results['S1_shape'] = False

# --- Check S2: param count in [730M, 770M] ---
n = model.get_num_params()
if 730_000_000 <= n <= 770_000_000:
    print(f"[PASS] S2 params: {n:,} ({n / 1e6:.1f}M)")
    results['S2_params'] = True
else:
    print(f"[FAIL] S2 params: {n:,} outside [730M, 770M]")
    results['S2_params'] = False

# --- Check S3: initial CE loss in [10.0, 11.5] ---
torch.manual_seed(42)
losses_s3 = []
with torch.no_grad():
    for _ in range(10):
        toks = torch.randint(0, cfg.vocab_size, (4, 65), device=device)
        inp, tgt = toks[:, :-1], toks[:, 1:]
        logits_s3, _ = model(inp)
        l = F.cross_entropy(
            logits_s3.float().reshape(-1, cfg.vocab_size),
            tgt.reshape(-1),
            ignore_index=-100,
        )
        losses_s3.append(l.item())

mean_loss_s3 = sum(losses_s3) / len(losses_s3)
theoretical = math.log(cfg.vocab_size)
if 10.0 <= mean_loss_s3 <= 11.5:
    print(f"[PASS] S3 initial loss: {mean_loss_s3:.4f}  (theoretical≈{theoretical:.4f})")
    results['S3_loss'] = True
else:
    print(f"[FAIL] S3 initial loss: {mean_loss_s3:.4f} outside [10.0, 11.5]")
    results['S3_loss'] = False

# --- Check S4: seq_len guard ---
guard_passed = False
try:
    bad_tok = torch.zeros(1, cfg.max_seq_len + 1, dtype=torch.long, device=device)
    _ = model(bad_tok)
    print("[FAIL] S4 seq_len guard: no error raised")
except ValueError:
    print("[PASS] S4 seq_len guard: ValueError raised correctly")
    guard_passed = True
except Exception as e:
    print(f"[FAIL] S4 seq_len guard: unexpected exception: {e}")
results['S4_guard'] = guard_passed

print("\nSanity summary:", results)

---
## Cell 5: Single-Batch Overfit (50 AdamW Steps, Loss → < 0.1)

In [ ]:
# Cell 5 (Spec: Cell 4) — single-batch overfit loop
# 8 sequences × 512 tokens, 50 AdamW steps, lr=1e-3
# Constitution: bfloat16 for training, TF32 matmul enabled
import time

# Reset to training mode with bfloat16
model.train()

torch.manual_seed(0)
OVERFIT_B   = 8
OVERFIT_T   = 512
OVERFIT_LR  = 1e-3
OVERFIT_STEPS = 50
LOG_EVERY   = 10

# Fix a single batch
fixed_tokens = torch.randint(0, cfg.vocab_size, (OVERFIT_B, OVERFIT_T + 1), device=device)
overfit_inp = fixed_tokens[:, :-1]   # (8, 512)
overfit_tgt = fixed_tokens[:, 1:]    # (8, 512)

optimizer = torch.optim.AdamW(model.parameters(), lr=OVERFIT_LR, fused=True)

loss_log = []
t0 = time.time()

for step in range(1, OVERFIT_STEPS + 1):
    optimizer.zero_grad()

    logits_ov, _ = model(overfit_inp)
    loss_ov = F.cross_entropy(
        logits_ov.float().reshape(-1, cfg.vocab_size),
        overfit_tgt.reshape(-1),
    )
    loss_ov.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % LOG_EVERY == 0 or step == 1:
        elapsed = time.time() - t0
        print(f"step {step:3d}/{OVERFIT_STEPS}  loss={loss_ov.item():.4f}  elapsed={elapsed:.1f}s")
        loss_log.append({'step': step, 'loss': loss_ov.item()})

final_overfit_loss = loss_ov.item()
total_time = time.time() - t0

if final_overfit_loss < 0.1:
    print(f"\n[PASS] Overfit: step-50 loss={final_overfit_loss:.4f} < 0.1 in {total_time:.1f}s")
else:
    print(f"\n[FAIL] Overfit: step-50 loss={final_overfit_loss:.4f} >= 0.1 (target < 0.1)")

---
## Cell 6: Clean-Batch Inference Check

In [ ]:
# Cell 6 (Spec: Cell 5) — clean-batch inference check
# After overfitting, the model should memorise the fixed batch
# but a fresh unseen batch should produce loss near ln(64000) ≈ 11.07
model.eval()

torch.manual_seed(99)
clean_tokens = torch.randint(0, cfg.vocab_size, (OVERFIT_B, OVERFIT_T + 1), device=device)
clean_inp = clean_tokens[:, :-1]
clean_tgt = clean_tokens[:, 1:]

with torch.no_grad():
    logits_clean, _ = model(clean_inp)
    clean_loss = F.cross_entropy(
        logits_clean.float().reshape(-1, cfg.vocab_size),
        clean_tgt.reshape(-1),
    ).item()

theoretical_clean = math.log(cfg.vocab_size)
print(f"Clean-batch loss : {clean_loss:.4f}")
print(f"Theoretical (ln({cfg.vocab_size})): {theoretical_clean:.4f}")

# Expect clean loss to remain near theoretical maximum-entropy baseline
# The model overfits the training batch but has not generalised
if clean_loss >= theoretical_clean - 2.0:
    print(f"[PASS] Clean-batch loss ({clean_loss:.4f}) is near theoretical ({theoretical_clean:.4f}) — no leakage")
else:
    print(f"[WARN] Clean-batch loss ({clean_loss:.4f}) is unexpectedly low — check for data leakage")

---
## Cell 7: Results Summary

In [ ]:
# Cell 7 (Spec: Cell 6) — results summary table
from datetime import datetime

print("=" * 60)
print("Phase 2 — Model Architecture: Results Summary")
print(f"Timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Device    : {gpu_name}")
print(f"Dtype     : {'bfloat16' if USE_BF16 else 'float32'}")
print("=" * 60)

summary_rows = [
    ("Model instantiation",       "PASS"),
    ("Param count (749.5M ± 20M)", "PASS" if results.get('S2_params') else "FAIL"),
    ("Forward shape (2,128)→(2,128,64000)", "PASS" if results.get('S1_shape') else "FAIL"),
    ("Initial CE loss ∈ [10.0, 11.5]",     "PASS" if results.get('S3_loss') else "FAIL"),
    ("seq_len guard (ValueError)",          "PASS" if results.get('S4_guard') else "FAIL"),
    ("Overfit step-50 loss < 0.1",          "PASS" if final_overfit_loss < 0.1 else "FAIL"),
    (f"Clean-batch loss ≈ {theoretical_clean:.2f}", "PASS" if clean_loss >= theoretical_clean - 2.0 else "WARN"),
]

print(f"{'Check':<45} {'Status'}")
print("-" * 55)
all_pass = True
for label, status in summary_rows:
    marker = "✓" if status == "PASS" else ("⚠" if status == "WARN" else "✗")
    print(f"{label:<45} {marker} {status}")
    if status == "FAIL":
        all_pass = False

print("=" * 60)
if all_pass:
    print("ALL CHECKS PASSED — Epic 2 (Model Architecture) complete")
else:
    print("SOME CHECKS FAILED — Review output above")